# Chapter 5 — Loading and Storing Data

Part 2: Pointer arithmetic and memory operations.

**Ladder.**
- **Method 01:** host mask with `torch.where` and a padding buffer.
- **Method 02:** masked `tl.load` / `tl.store` with scalar and vector tiles.
- **Method 03:** read-modify-write and fused masked reductions.

**Thread.**
- Offsets first: `pid * BLOCK + tl.arange(0, BLOCK)` fixes the lane map.
- The mask `offs < N` decides which lanes exist and guards both memory doors.
- `other` gives non-existent lanes the identity of the next reduction.
- The same four objects — offsets, mask, value, `other` — recur in copy, add, RMW, and reduction kernels.
- The address pattern decides what those lanes cost: contiguous coalesces, strided multiplies transactions.

**Study tip.**
- Write the load/store table for $N=10$, `BLOCK=4` before writing code.
- List the three program ids, the four offsets each owns, and the tail mask.
- Choose `other` from the operation that follows the load: $0$ for sum, $1$ for product, $-\infty$ for max.

**Navigation:** [Index](../../00_index.ipynb) · [← Chapter 4](../ch04_pointers/04_complete.ipynb) · [Chapter 6 →](../ch06_indexing/06_complete.ipynb)

**Estimated time:** 70–90 min · **Difficulty:** Intermediate

**Learning objectives**
- Write `tl.load(ptr + offs, mask=..., other=...)` and `tl.store(ptr + offs, value, mask=...)`, and say which arguments are tiles and which are scalars.
- Compute the program count $P=\lceil N/\text{BLOCK}\rceil$ and the exact 1-D and 2-D masks, including the tail tile.
- Choose the neutral `other` for sum, product, max, min, logarithm, and a denominator; explain the failure each wrong choice causes.
- Guard the load **and** the store so no lane ever addresses memory outside the allocation.
- Use scalar loads, vector loads, multiple input/output pointers, and dtype casts.
- Recognise a read-modify-write and a coalesced access pattern.

**Prerequisites:** Chapter 4 (pointers, element offsets, strides), Chapter 3 (`tl.arange`, `tl.zeros`, broadcasting), and Chapter 2 (program ids and launch grids).

**Book apparatus:** [Setup](../../docs/SETUP.md) · [Notation](../../docs/NOTATION.md) · [Glossary](../../docs/GLOSSARY.md) · [API index](../../docs/API_INDEX.md) · [Formula index](../../docs/FORMULA_INDEX.md) · [Diagrams](../../docs/DIAGRAMS.md) · [Troubleshooting](../../docs/TROUBLESHOOTING.md) · [Exercise hints](../../docs/SOLUTIONS.md)

## Concept diagrams

**Connection.** Every diagram below uses the Chapter 4 address map and the chapter spine $N=10$, `BLOCK=4`.

### The load path

```text
logical ids      pid = 2, BLOCK = 4
                   |
offs = pid*BLOCK + tl.arange(0, BLOCK)      [8 9 10 11]
                   |
ptr  = x_ptr + offs                          [addresses]
mask = offs < N                              [1 1 0 0]
                   |
x = tl.load(ptr, mask=mask, other=0.0)       [9 10 0 0]
```

### The store path

```text
value tile y (registers)  [ ... ]
        |
ptr  = out_ptr + offs
mask = offs < N
        |
tl.store(ptr, y, mask=mask)   -> only mask==1 lanes write
```

### The masked tile

```text
offs   0  1  2  3 | 4  5  6  7 | 8  9 10 11
mask   1  1  1  1 | 1  1  1  1 | 1  1  0  0
x     -1 -2 -3 -4 |-5 -6 -7 -8 |-9 -10 o  o
                                      ^^^^ other
```

### `other` is the neutral element of the next operation

| Next operation | Neutral `other` | Effect on a masked lane |
|---|---|---|
| sum | `0.0` | adds nothing |
| product | `1.0` | multiplies by one |
| max | `-inf` | can never win |
| min | `+inf` | can never win |
| `log` then sum | `1.0` | `log(1) = 0` adds nothing |
| denominator | `1.0` | never divides by zero |

- **Rule.** One lane owns one element of the compile-time tile.
- **Rule.** The mask is a predicate tile, not a branch.
- **Rule.** `other` fills false lanes only on a load; a masked store skips the write.
- **Rule.** The mask answers which lanes exist.
- **Rule.** `other` answers what the non-existent lanes contribute.
- **Failure.** Choosing `other` for the data instead of the consumer biases a reduction.
- **Bridge.** §5.1 and §5.2 make the two paths precise.

In [ ]:
import sys
from pathlib import Path

root = Path.cwd()
if (root / "00_Basic").exists():
    root = root / "00_Basic"
while root.name != "00_Basic" and root.parent != root:
    root = root.parent
sys.path.insert(0, str(root))

import math

import pytest
import torch
import triton
import triton.language as tl

from ttools import DEVICE, HAS_GPU, check, require_gpu, run_if_gpu

print("device:", DEVICE, "| GPU:", HAS_GPU, "| triton:", triton.__version__,
      "| torch:", torch.__version__)

## Part I — Mathematics and concepts

**Connection.** Chapter 4 built the address $p_i=\text{base}+\text{offs}_i$; this part adds the two operations that read and write it.

**Rule.** A Triton program touches global memory only through `tl.load` and `tl.store`.
**Rule.** Everything else — arithmetic, comparisons, reductions — happens in registers.
**Rule.** Shapes are fixed by the tile, not by the runtime tensor: `tl.arange(0, BLOCK)` has length `BLOCK` at compile time.

**Formula.** A load is a masked gather and a store is a masked scatter:

$$
x_i=\begin{cases}\operatorname{load}(p_i),&m_i=1\\ \text{other},&m_i=0\end{cases}
\qquad
\operatorname{store}(p_i,y_i,m_i)=\begin{cases}y_i,&m_i=1\\ \text{skip},&m_i=0\end{cases}
$$

**Triton.** The three quantities that must line up are:

| Quantity | Shape | Produced by |
|---|---|---|
| offsets | `(BLOCK,)` or `(BLOCK_M, BLOCK_N)` | `pid*BLOCK + tl.arange(...)` |
| mask | same as offsets | `offs < n_elements` |
| value | same as offsets | `tl.load(..., other=...)` |

**Rule.** `other` is a scalar, broadcast to every masked-out lane; it does not change the tile shape.
**Failure.** A tile whose offsets, mask, and value disagree on length fails at compile time or writes the wrong lanes.
**Bridge.** §5.1 begins with the read door.

### 5.1 `tl.load()`

**Connection.** Builds on Chapter 4's pointer arithmetic: `x_ptr + offs` is a tile of addresses.

**Rule.** With no mask, every lane reads.
**Rule.** With a mask, only true lanes access memory; false lanes receive `other`.
**Formula.** $x_i=\operatorname{load}(p_i)$ if $m_i=1$, and $x_i=\text{other}$ if $m_i=0$.
**Formula.** The result has the shape of `offs`; a scalar offset gives shape `()`.
**Triton.** `x = tl.load(x_ptr + offs, mask=mask, other=0.0)`
**Failure.** `tl.load` is not bounds-checked; a true lane past the allocation is undefined behaviour.
**Bridge.** The store is the mirror operation.

### 5.2 `tl.store()`

**Connection.** Mirrors §5.1 with the same offsets and mask, now on the write side.

**Rule.** False lanes are not written; their previous contents survive.
**Rule.** There is no `other` on a store.
**Formula.** $\operatorname{store}(p_i,y_i,m_i)=y_i$ if $m_i=1$, otherwise the lane is skipped.
**Formula.** Value, pointer, and mask share the tile shape; a scalar value broadcasts.
**Triton.** `tl.store(out_ptr + offs, value, mask=mask)`
**Failure.** If the store is wider than the allocation, the kernel corrupts memory even when the load was masked.
**Bridge.** §5.3 isolates the two special cases of the load: scalar and vector.

In [ ]:
x = torch.arange(1.0, 11.0)          # N = 10
N, BLOCK = x.numel(), 4
pid = 2
offs = pid * BLOCK + torch.arange(BLOCK)
mask = offs < N
safe = offs.clamp(max=N - 1)          # host stand-in for an address the mask suppresses
loaded = torch.where(mask, x[safe], torch.zeros((), dtype=x.dtype))

out = torch.full((12,), -1.0)
out[offs[mask]] = loaded[mask]

assert (N, BLOCK, pid) == (10, 4, 2)
assert triton.cdiv(N, BLOCK) == 3
assert offs.tolist() == [8, 9, 10, 11]
assert mask.tolist() == [True, True, False, False]
assert loaded.tolist() == [9.0, 10.0, 0.0, 0.0]
assert out.tolist() == [-1.0] * 8 + [9.0, 10.0] + [-1.0, -1.0]
assert out[8:10].tolist() == [9.0, 10.0]
assert out[10:].tolist() == [-1.0, -1.0]

### 5.3 Scalar load

**Connection.** Narrows §5.1 to the case where `offs` is a single address.

**Rule.** A scalar load produces a rank-0 value.
**Rule.** One transaction per program is cheaper than one load per lane when the value is reused.
**Formula.** $v=\operatorname{load}(p)$ and $v=\operatorname{load}(p+i)$ both have shape `()`.
**Where.** Scale factors, epsilon, clamp bounds, row pointers.
**Triton.** `v = tl.load(ptr + i)`
**Failure.** The address must be valid; a scalar load has no tail unless you pass a mask.
**Bridge.** §5.4 replaces the single address with a tile.

### 5.4 Vector load

**Connection.** Generalises §5.3 from one address to a `BLOCK`-lane offset tile.

**Rule.** The result length is the compile-time `BLOCK`.
**Formula.** $x_i=\operatorname{load}(p_i)$ for $i=0,\ldots,B-1$.
**Where.** Every elementwise kernel in the rest of Part 2.
**Triton.** `offs = pid * BLOCK + tl.arange(0, BLOCK); x = tl.load(x_ptr + offs)`
**Rule.** `shape(tl.load(p)) = shape(p)`; a `(BLOCK_M, BLOCK_N)` pointer tile returns a `(BLOCK_M, BLOCK_N)` value tile.
**Failure.** Strided offsets still load the same count but may touch one cache line per lane.
**Bridge.** §5.5 adds the mask that makes the tail tile safe.

In [ ]:
x = torch.arange(12.0)
scalar = x[3]
tile = x[4:8]

assert scalar.shape == () and scalar.item() == 3.0
assert tile.shape == (4,) and tile.tolist() == [4.0, 5.0, 6.0, 7.0]
assert (scalar + tile).tolist() == [7.0, 8.0, 9.0, 10.0]
assert (tile + tile).tolist() == [8.0, 10.0, 12.0, 14.0]

try:
    x[12]
except IndexError:
    pass
else:
    raise AssertionError("host indexing should reject an out-of-range scalar")

# A 2-D pointer tile keeps the shape of its offsets and mask.
offs2 = torch.arange(2)[:, None] * 3 + torch.arange(3)[None, :]
mask2 = offs2 < 6
assert offs2.shape == mask2.shape == (2, 3)
assert mask2.tolist() == [[True, True, True], [True, True, True]]

### 5.5 Masked load

**Connection.** Adds the `offs < N` predicate to the §5.4 vector load.

**Rule.** True lanes read; false lanes read nothing and receive `other`.
**Formula.** $x_i=\operatorname{load}(p_i)$ if $m_i=1$, otherwise `other`.
**Example.** For $N=10$, `BLOCK=4`, the last program owns offsets $8,9,10,11$ and only $8,9$ are real.
**Triton.** `x = tl.load(x_ptr + offs, mask=offs < N, other=0.0)`
**Failure.** A true lane past the allocation faults even if the store is masked.
**Bridge.** §5.6 gives the store the same treatment.

### 5.6 Masked store

**Connection.** Mirrors §5.5 on the write side.

**Rule.** Only true lanes write; false lanes leave the destination untouched.
**Formula.** $y_i$ is written if $m_i=1$, otherwise the write is skipped.
**Example.** `tl.store(out_ptr + offs, tl.where(mask, y, 0.0), mask=mask)` has the same memory effect as `tl.store(..., mask=mask)`.
**Triton.** `tl.store(out_ptr + offs, y, mask=offs < N)`
**Failure.** Dropping the store mask writes zeros or garbage into the tail; for a tensor of length $N$ it writes past the end.
**Bridge.** §5.7 explains what `other` must contain.

In [ ]:
x = torch.arange(10.0)
offs = torch.arange(8, 12)
mask = offs < 10
loaded = torch.where(mask, x[offs.clamp(max=9)], torch.tensor(-1.0))
dest = torch.full((12,), 99.0)
dest[offs[mask]] = loaded[mask]

assert offs.tolist() == [8, 9, 10, 11]
assert mask.tolist() == [True, True, False, False]
assert loaded.tolist() == [8.0, 9.0, -1.0, -1.0]
assert dest[8:10].tolist() == [8.0, 9.0]
assert dest[offs[~mask]].tolist() == [99.0, 99.0]
assert int(mask.sum()) == 2 and int((~mask).sum()) == 2
assert dest.tolist() == [99.0] * 8 + [8.0, 9.0] + [99.0] * 2

### 5.7 The `other` parameter

**Connection.** Supplies the value that §5.5 gives to masked-out load lanes.

**Rule.** `other` is not a clamp and not a default for missing data; it is the identity of the next consumer.
**Formula.** $x_i=\operatorname{load}(p_i)$ if $m_i=1$, otherwise `other`; `other` is a scalar broadcast to the tile.

| Consumer | `other` | Why |
|---|---|---|
| `tl.sum` | `0.0` | $x+0=x$ |
| product | `1.0` | $x\cdot1=x$ |
| `tl.max` | `-inf` | it can never be the maximum |
| `tl.min` | `+inf` | it can never be the minimum |
| `tl.log` then sum | `1.0` | $\log 1=0$ |
| denominator | `1.0` | never $0/0$ |
| store-only copy | `0.0` | value is discarded by the store mask |

**Example.** For the tile $(-9,-10)$ with two masked lanes:

| Operation | `other` | padded tile | result |
|---|---|---|---|
| sum | $0$ | $(-9,-10,0,0)$ | $-19$ |
| product | $1$ | $(-9,-10,1,1)$ | $90$ |
| max | $-\infty$ | $(-9,-10,-\infty,-\infty)$ | $-9$ |
| min | $+\infty$ | $(-9,-10,+\infty,+\infty)$ | $-10$ |

**Rule.** The wrong choices are visible: `max` with `other=0.0` returns $0$ instead of $-9$; `sum` with `other=-inf` returns $-\infty$ instead of $-19$.
**Triton.** `tl.load(ptr + offs, mask=mask, other=-float("inf"))`
**Failure.** A wrong `other` is silent: mask and shapes are right, the reduction is wrong.
**Bridge.** §5.8 computes the boundary mask for 1-D and 2-D tiles.

In [ ]:
tile = torch.tensor([-9.0, -10.0])

summed = torch.cat([tile, torch.zeros(2)]).sum().item()
product = torch.cat([tile, torch.full((2,), 1.0)]).prod().item()
maximum = torch.cat([tile, torch.full((2,), -math.inf)]).max().item()
minimum = torch.cat([tile, torch.full((2,), math.inf)]).min().item()

assert summed == -19.0
assert product == 90.0
assert maximum == -9.0
assert minimum == -10.0
assert torch.cat([tile, torch.zeros(2)]).max().item() == 0.0
assert torch.cat([tile, torch.full((2,), -math.inf)]).sum().item() == -math.inf
print("neutral elements verified")

### 5.8 Boundary checking

**Connection.** Generalises the 1-D tail mask from §5.5 to a 2-D tile.

**Rule.** A 1-D boundary is $m_i=[\text{offs}_i\lt N]$.
**Formula.** A 2-D boundary is $m_{ij}=[r_i\lt M]\,\&\,[c_j\lt N]$, with $r_i=p_mB_M+i$ and $c_j=p_nB_N+j$.
**Example.** For $M=3$, $N=5$, $p_m=p_n=1$, `BLOCK_M=2`, `BLOCK_N=4`, rows are $2,3$ and columns are $4,5,6,7$; only lane $(2,4)$ is active.
**Where.** Every tile whose `BLOCK` does not divide the tensor dimension.
**Triton.** `mask = (rows[:, None] < M) & (cols[None, :] < N)`
**Rule.** The comparison runs on every lane; only the memory access is suppressed.
**Bridge.** §5.9 turns the mask into the safety mechanism.

### 5.9 Out-of-bounds prevention

**Connection.** Applies the §5.8 mask to both memory doors.

**Rule.** The mask must guard the load and the store, and every program must compute the same mask.
**Failure.** A missing load mask reads garbage or faults; a missing store mask corrupts a neighbour buffer; neither raises a Python exception.
**Rule.** The host-side `IndexError` from `x[12]` does not exist inside a kernel.
**Where.** Every kernel in Part III and Part IV.
**Triton.** Use one predicate for `tl.load` and `tl.store`.
**Bridge.** §5.10 adds the dtype dimension to the byte count.

In [ ]:
N, BLOCK = 10, 4
P = triton.cdiv(N, BLOCK)

assert (N, BLOCK, P) == (10, 4, 3)
assert P * BLOCK == 12 and P * BLOCK - N == 2

tiles = []
for pid in range(P):
    offs = pid * BLOCK + torch.arange(BLOCK)
    tiles.append((offs.tolist(), (offs < N).tolist()))
assert tiles == [
    ([0, 1, 2, 3], [True, True, True, True]),
    ([4, 5, 6, 7], [True, True, True, True]),
    ([8, 9, 10, 11], [True, True, False, False]),
]

M, Nc, BM, BN = 3, 5, 2, 4
rm = 1 * BM + torch.arange(BM)
cn = 1 * BN + torch.arange(BN)
mask2 = (rm[:, None] < M) & (cn[None, :] < Nc)
assert rm.tolist() == [2, 3] and cn.tolist() == [4, 5, 6, 7]
assert mask2.shape == (2, 4)
assert mask2.int().tolist() == [[1, 0, 0, 0], [0, 0, 0, 0]]
assert int(mask2.sum()) == 1 and mask2.numel() == 8

try:
    torch.arange(10.0)[10]
except IndexError:
    pass
else:
    raise AssertionError("host indexing should reject an out-of-range scalar")

### 5.10 Load and store dtypes

**Connection.** Adds the element width to the §5.6 masked store and the §5.5 masked load.

**Rule.** The pointer's element type decides transaction width and interpretation.

| dtype | bytes/element | read + write for $N=10$ |
|---|---|---|
| `float64` | 8 | 160 B |
| `float32` | 4 | 80 B |
| `float16` / `bfloat16` | 2 | 40 B |
| `int8` | 1 | 20 B |

**Formula.** $\text{bytes}=N_{\text{active}}(b_{\text{load}}+b_{\text{store}})$.
**Rule.** `tl.load` returns a tile of the pointer dtype; `float16` arithmetic stays in `float16` unless cast.
**Triton.** `x = tl.load(x_ptr + offs, mask=mask, other=0.0).to(tl.float32)`
**Rule.** `other` must be representable in the loaded dtype; `other=-inf` needs a floating dtype.
**Failure.** A wider value stored through a narrow pointer truncates silently.
**Bridge.** §5.11 and §5.12 add more pointers.

In [ ]:
traffic = {
    torch.float64: 160,
    torch.float32: 80,
    torch.float16: 40,
    torch.bfloat16: 40,
    torch.int8: 20,
}
for dt, expected_bytes in traffic.items():
    t = torch.zeros(10, dtype=dt)
    assert t.element_size() in (1, 2, 4, 8)
    assert 2 * 10 * t.element_size() == expected_bytes

x16 = torch.tensor([1.5, -2.25, 3.0], dtype=torch.float16)
assert x16.tolist() == [1.5, -2.25, 3.0]
assert x16.float().tolist() == [1.5, -2.25, 3.0]
assert (x16.float() * 2.0).half().tolist() == [3.0, -4.5, 6.0]
assert torch.tensor([3.9]).to(torch.int8).tolist() == [3]
print("dtype traffic verified")

### 5.11 Multiple input pointers

**Connection.** Repeats the §5.4 vector load once per input buffer.

**Rule.** Each load has its own address expression and, if lengths differ, its own mask.
**Formula.** $y_i=a_i x_i+b_i$.
**Triton.** `a = tl.load(a_ptr + offs, mask=mask, other=0.0)` and the same for `b`.
**Failure.** Two loads per output element double input traffic; fusing the producer is the alternative.
**Bridge.** §5.12 adds a second output.

### 5.12 Multiple output pointers

**Connection.** Mirrors §5.11: several stores instead of several loads.

**Rule.** Each extra store adds $N b$ bytes of write traffic.
**Formula.** $(y_i,s_i)=f(x_i)$.
**Where.** Saving an auxiliary output for a backward pass or a later kernel.
**Triton.** `tl.store(y_ptr + offs, y, mask=mask)` then `tl.store(s_ptr + offs, s, mask=mask)`.
**Failure.** An output that aliases an input is a read-modify-write; only do it deliberately.
**Bridge.** §5.13 makes the aliasing case explicit.

In [ ]:
a = torch.tensor([1.0, 2.0, 3.0])
x = torch.tensor([4.0, 5.0, 6.0])
b = torch.tensor([7.0, 8.0, 9.0])
y = a * x + b
s = x * x

assert y.tolist() == [11.0, 18.0, 27.0]
assert s.tolist() == [16.0, 25.0, 36.0]
assert 3 * x.numel() == 9 and 2 * x.numel() == 6
assert 5 * x.numel() == 15
assert 4 * x.numel() == 12 and x.numel() == 3

### 5.13 Read-modify-write

**Connection.** Combines §5.2 (store) with §5.11 (the same buffer as input and output).

**Rule.** Load and store the same address with the same offsets and mask.
**Formula.** $y_i\leftarrow y_i+x_i$.
**Rule.** Within one program the order is guaranteed; across programs it is not.
**Failure.** Two programs updating one address race and the last store wins.
**Triton.** `y = tl.load(y_ptr + offs, mask=mask, other=0.0); tl.store(y_ptr + offs, y + x, mask=mask)`
**Where.** In-place scale, in-place add, and optimizer updates.
**Rule.** Cross-program accumulation needs `tl.atomic_add` (Chapter 21).
**Bridge.** §5.14 asks what the address pattern costs.

In [ ]:
y = torch.tensor([1.0, 2.0, 3.0, 4.0])
x = torch.tensor([10.0, 20.0, 30.0, 40.0])
before = y.clone()
y.add_(x)

assert before.tolist() == [1.0, 2.0, 3.0, 4.0]
assert x.tolist() == [10.0, 20.0, 30.0, 40.0]
assert y.tolist() == [11.0, 22.0, 33.0, 44.0]
assert 3 * y.numel() == 12

### 5.14 Memory access patterns

**Connection.** Applies §5.4's vector load to the address order chosen by the offsets.

**Rule.** The hardware fetches aligned segments; a warp is coalesced when consecutive lanes touch consecutive addresses.
**Formula.** $\text{offs}_i=\text{base}+i\Rightarrow\Delta p=1$ element.
**Example.** For $A=\begin{bmatrix}0&1&2&3\\4&5&6&7\\8&9&10&11\end{bmatrix}$, shape $(3,4)$, stride $(4,1)$, the two offset maps are

$$
\text{row-major}=\begin{bmatrix}0&1&2&3\\4&5&6&7\\8&9&10&11\end{bmatrix},
\qquad
\text{column-major}=\begin{bmatrix}0&3&6&9\\1&4&7&10\\2&5&8&11\end{bmatrix}.
$$

**Rule.** For a $32\times32$ `float32` matrix, 32 contiguous lanes span $31\cdot4=124$ B and touch one 128 B line; a column step is $32\cdot4=128$ B, so 32 lanes span $31\cdot128+4=3972$ B and touch 32 lines, with only $128/4096=1/32$ useful.
**Where.** Tile copies, transposes, and GEMM.
**Triton.** `offs = rows[:, None] * stride_m + cols[None, :] * stride_n`
**Failure.** Column-major access multiplies transactions without changing element count.
**Bridge.** The chapter map collects the rules; the worked example checks the spine.
**Where this leads.** Chapter 6 turns these 1-D offsets into 2-D and 3-D index maps, including tiled and strided addressing.

In [ ]:
A = torch.arange(12.0).reshape(3, 4)
T = A.T
assert A.shape == (3, 4) and A.stride() == (4, 1) and A.is_contiguous()
assert A[1].tolist() == [4.0, 5.0, 6.0, 7.0]
assert A[1].stride() == (1,) and A[1].is_contiguous()
assert A[:, 1].tolist() == [1.0, 5.0, 9.0]
assert A[:, 1].stride() == (4,) and not A[:, 1].is_contiguous()
assert T.shape == (4, 3) and T.stride() == (1, 4) and not T.is_contiguous()
assert T.contiguous().stride() == (3, 1)
assert T.contiguous().data_ptr() != A.data_ptr()

rows, cols = torch.arange(3), torch.arange(4)
row_off = rows[:, None] * 4 + cols[None, :]
col_off = rows[:, None] + cols[None, :] * 3
assert row_off.tolist() == [[0, 1, 2, 3], [4, 5, 6, 7], [8, 9, 10, 11]]
assert col_off.tolist() == [[0, 3, 6, 9], [1, 4, 7, 10], [2, 5, 8, 11]]
assert (row_off[0, 1:] - row_off[0, :-1]).tolist() == [1, 1, 1]
assert (col_off[1:, 0] - col_off[:-1, 0]).tolist() == [1, 1]

B = torch.arange(32 * 32, dtype=torch.float32).reshape(32, 32)
row_bytes = 32 * B.element_size()
col_bytes = 31 * B.stride(0) * B.element_size() + B.element_size()
assert row_bytes == 128 and triton.cdiv(row_bytes, 128) == 1
assert col_bytes == 3972 and triton.cdiv(col_bytes, 128) == 32
assert row_bytes / (triton.cdiv(col_bytes, 128) * 128) == 1 / 32
print("stride and coalescing arithmetic verified")

## Chapter map

**Connection.** Collects the rule, shape constraint, and kernel for each section.

| § | Topic | Key formula/rule | Shape/mask rule | Kernel/demo |
|---|---|---|---|---|
| **5.1** | `tl.load()` | $x_i=\operatorname{load}(p_i)$ if $m_i$, else `other` | `shape(x) = shape(offs)` | masked copy (Method 02) |
| **5.2** | `tl.store()` | write $y_i$ only if $m_i$ | `shape(value) = shape(offs)` | masked copy (Method 02) |
| **5.3** | Scalar load | $v=\operatorname{load}(p)$ | result shape `()` | scalar scale (Method 02) |
| **5.4** | Vector load | $x_i=\operatorname{load}(p_i)$ | `(BLOCK,)` or `(BM,BN)` | masked copy (Method 02) |
| **5.5** | Masked load | $x_i=\operatorname{load}(p_i)$ if $m_i$, else `other` | mask shape = offset shape | copy with `other=0.0` |
| **5.6** | Masked store | $y_i$ written only if $m_i$ | store mask = load mask | sentinel-tail test |
| **5.7** | The `other` parameter | identity of the next reduction | scalar, broadcast to the tile | neutral-element table |
| **5.8** | Boundary checking | $m_i=[\text{offs}_i\lt N]$, $m_{ij}=[r_i\lt M]\,\&\,[c_j\lt N]$ | outer product for 2-D | 1-D and 2-D tail masks |
| **5.9** | Out-of-bounds prevention | mask the load **and** the store | same predicate on both doors | workthrough of all three programs |
| **5.10** | Load and store dtypes | bytes $=N_{\text{active}}(b_{\text{load}}+b_{\text{store}})$ | value dtype = pointer dtype | dtype traffic table |
| **5.11** | Multiple input pointers | $y_i=a_i x_i+b_i$ | one mask per input length | two-input add (Method 02) |
| **5.12** | Multiple output pointers | $(y_i,s_i)=f(x_i)$ | one store per output | two-output host demo |
| **5.13** | Read-modify-write | $y_i\leftarrow y_i+x_i$ | same offsets and mask on both doors | in-place add (Method 03) |
| **5.14** | Memory access patterns | coalesced if $\Delta p=1$ across lanes | last axis contiguous | stride experiment + column-major copy |
| **★** | Worked example | $N=10$, `BLOCK=4`, $P=3$ | tail mask `T T F F` | `test_worked_example` |

## Worked numerical example — $N=10$, `BLOCK=4`

**Connection.** Applies §5.7 (`other`) and §5.8 (tail mask) to the chapter spine.

**Example.** Take $x=(-1,-2,-3,-4,-5,-6,-7,-8,-9,-10)$; then $P=\lceil 10/4\rceil=3$, with 12 declared lanes and 10 real elements.

| pid | offs | mask | lanes | load, `other=0.0` | load, `other=-inf` |
|---|---|---|---|---|---|
| 0 | $0,1,2,3$ | $\mathrm{T,T,T,T}$ | 4 | $(-1,-2,-3,-4)$ | $(-1,-2,-3,-4)$ |
| 1 | $4,5,6,7$ | $\mathrm{T,T,T,T}$ | 4 | $(-5,-6,-7,-8)$ | $(-5,-6,-7,-8)$ |
| 2 | $8,9,10,11$ | $\mathrm{T,T,F,F}$ | 2 | $(-9,-10,0,0)$ | $(-9,-10,-\infty,-\infty)$ |

**Formula.**

$$
S_0=(-10)+(-26)+(-19)=-55,\quad S_{-\infty}=-\infty,\quad
M_{-\infty}=-1,\quad M_0=0.
$$

**Rule.** `other=0.0` is right for the sum and wrong for the max; `other=-inf` is right for the max and destroys the sum.
**Failure.** The mask and the shapes are correct in both cases; only `other` decides which answer is right.
**Triton.** The next cell repeats the table with `torch.where` and asserts every row.
**Bridge.** Part II replaces `torch.where` with a padding buffer and a CPU test.

In [ ]:
x = -torch.arange(1.0, 11.0)
N, BLOCK = x.numel(), 4
P = triton.cdiv(N, BLOCK)
assert (N, BLOCK, P) == (10, 4, 3)

zero_rows, inf_rows, offs_rows, mask_rows = [], [], [], []
for pid in range(P):
    offs = pid * BLOCK + torch.arange(BLOCK)
    mask = offs < N
    z = torch.where(mask, x[offs.clamp(max=N - 1)], torch.zeros(BLOCK))
    n = torch.where(mask, x[offs.clamp(max=N - 1)], torch.full((BLOCK,), -math.inf))
    offs_rows.append(offs)
    mask_rows.append(mask)
    zero_rows.append(z)
    inf_rows.append(n)

assert [o.tolist() for o in offs_rows] == [
    [0, 1, 2, 3], [4, 5, 6, 7], [8, 9, 10, 11],
]
assert [m.tolist() for m in mask_rows] == [
    [True, True, True, True],
    [True, True, True, True],
    [True, True, False, False],
]
assert zero_rows[2].tolist() == [-9.0, -10.0, 0.0, 0.0]
assert inf_rows[2].tolist() == [-9.0, -10.0, -math.inf, -math.inf]

total_zero = sum(r.sum().item() for r in zero_rows)
total_inf = sum(r.sum().item() for r in inf_rows)
best_inf = max(r.max().item() for r in inf_rows)
best_zero = max(r.max().item() for r in zero_rows)
assert total_zero == -55.0
assert math.isinf(total_inf) and total_inf < 0
assert best_inf == -1.0
assert best_zero == 0.0 and best_zero != x.max().item()


def test_worked_example() -> None:
    assert [m.tolist() for m in mask_rows] == [
        [True, True, True, True],
        [True, True, True, True],
        [True, True, False, False],
    ]
    assert zero_rows[2].tolist() == [-9.0, -10.0, 0.0, 0.0]
    assert inf_rows[2].tolist() == [-9.0, -10.0, -math.inf, -math.inf]
    assert total_zero == -55.0
    assert math.isinf(total_inf) and total_inf < 0
    assert best_inf == -1.0
    assert best_zero == 0.0 and best_zero != x.max().item()


test_worked_example()
print("worked example verified")

## Part II — Method 01: host reference with `torch.where`

**Connection.** Starts from the §5.5–§5.7 rules but implements them eagerly, before any Triton.

**Rule.** Pad the flat tensor to `P*BLOCK` with the neutral value.
**Rule.** Mask with `offs < N`; `torch.where(mask, buf[offs], other)` is the masked load.
**Rule.** Store with `out = torch.where(mask, loaded, sentinel)`; false lanes keep the sentinel.

**Formula.**

$$
\text{pad}=(-N)\bmod\text{BLOCK},\qquad P=\left\lceil\frac{N}{\text{BLOCK}}\right\rceil.
$$

**Example.** For $N=10$, `BLOCK=4`, the padded buffer has 12 entries and the last mask row is `T T F F`.
**Where.** A CPU-only specification and a cross-check for the kernels.
**Triton.** The kernel does not need the padding because the mask suppresses the address.
**Failure.** Using `buf[offs]` without padding is an out-of-range host read.
**Bridge.** Part III replaces the host mask with `tl.load` / `tl.store`.

In [ ]:
def ref_tiles(x, BLOCK, other=0.0):
    """Method 01: pad to P*BLOCK, then mask with torch.where."""
    flat = x.reshape(-1)
    n = flat.numel()
    pad = (-n) % BLOCK
    buf = torch.cat([flat, torch.full((pad,), other, dtype=flat.dtype)])
    offs = torch.arange(buf.numel()).reshape(-1, BLOCK)
    mask = offs < n
    loaded = torch.where(mask, buf[offs], torch.full_like(buf[offs], other))
    return offs, mask, loaded


def ref_store(x, BLOCK, other=0.0, fill=-7.0):
    """Masked store: false lanes keep the sentinel `fill`."""
    offs, mask, loaded = ref_tiles(x, BLOCK, other)
    out = torch.full(offs.shape, fill, dtype=x.dtype)
    return torch.where(mask, loaded, out).reshape(-1)


def test_reference() -> None:
    x = -torch.arange(1.0, 11.0)
    offs, mask, loaded = ref_tiles(x, 4, 0.0)
    assert offs.tolist() == [[0, 1, 2, 3], [4, 5, 6, 7], [8, 9, 10, 11]]
    assert mask.tolist() == [[True] * 4, [True] * 4, [True, True, False, False]]
    assert loaded[2].tolist() == [-9.0, -10.0, 0.0, 0.0]
    out = ref_store(x, 4, 0.0)
    assert out[:10].tolist() == x.tolist()
    assert out[10:].tolist() == [-7.0, -7.0]
    assert ref_tiles(x, 4, -math.inf)[2][2].tolist() == [-9.0, -10.0, -math.inf, -math.inf]

    offs10, mask10, _ = ref_tiles(torch.arange(10.0), 4)
    assert offs10.tolist() == [[0, 1, 2, 3], [4, 5, 6, 7], [8, 9, 10, 11]]
    assert mask10.int().tolist() == [[1, 1, 1, 1], [1, 1, 1, 1], [1, 1, 0, 0]]


test_reference()
print("method 01 reference verified")

## Part III — Method 02: Triton load/store kernels

**Connection.** Turns the §5.5–§5.9 rules into the kernels the rest of the book uses.

**Rule.** The grid is 1-D with $P=\lceil N/\text{BLOCK}\rceil$ programs.
**Rule.** Program `pid` owns the half-open interval `[pid*BLOCK, (pid+1)*BLOCK)`, intersected with `[0, N)`.

| program | offsets | mask | lanes that touch memory |
|---|---|---|---|
| 0 | `0 1 2 3` | `T T T T` | 4 |
| 1 | `4 5 6 7` | `T T T T` | 4 |
| 2 | `8 9 10 11` | `T T F F` | 2 |

**Rule.** The tail tile is the only place the mask is false; the tests use `N=10` or `N=17` with `BLOCK=4`.
**Rule.** `other` follows the operation: the copy and the add use $0$; the reduction in Part IV uses $-\infty$ for max.
**Triton.** The generic skeleton is `offs = pid * BLOCK + tl.arange(0, BLOCK); mask = offs < n; x = tl.load(x_ptr + offs, mask=mask, other=0.0); tl.store(out_ptr + offs, x, mask=mask)`.
**Failure.** A kernel that forgets the mask fails on the last program, not on the first.
**Bridge.** Part IV adds aliasing and reductions.

In [ ]:
# Tile map for Part III: N=10, BLOCK=4, three programs.
N, BLOCK = 10, 4
assert triton.cdiv(N, BLOCK) == 3
assert [(pid * BLOCK + torch.arange(BLOCK)).tolist() for pid in range(3)] == [
    [0, 1, 2, 3], [4, 5, 6, 7], [8, 9, 10, 11],
]
assert [((pid * BLOCK + torch.arange(BLOCK)) < N).tolist() for pid in range(3)] == [
    [True, True, True, True],
    [True, True, True, True],
    [True, True, False, False],
]


@triton.jit
def m5_copy_kernel(x_ptr, out_ptr, n_elements, BLOCK: tl.constexpr):
    pid = tl.program_id(0)
    offs = pid * BLOCK + tl.arange(0, BLOCK)
    mask = offs < n_elements
    x = tl.load(x_ptr + offs, mask=mask, other=0.0)
    tl.store(out_ptr + offs, x, mask=mask)


def masked_copy(x, BLOCK=4, fill=-7.0):
    n = x.numel()
    p = triton.cdiv(n, BLOCK)
    buf = torch.full((p * BLOCK,), fill, dtype=x.dtype, device=x.device)
    m5_copy_kernel[(p,)](x, buf, n, BLOCK=BLOCK)
    return buf[:n].clone(), buf


def test_copy() -> None:
    require_gpu("m5 masked copy")
    x = torch.arange(10.0, device=DEVICE)
    got, buf = masked_copy(x, BLOCK=4)
    check("masked copy values", got, x)
    assert buf[10:].tolist() == [-7.0, -7.0]

    torch.manual_seed(0)
    xr = torch.randn(17, device=DEVICE)
    assert xr.numel() == 17
    got, _ = masked_copy(xr, BLOCK=4)
    check("masked copy, n=17", got, xr)


run_if_gpu(test_copy)

In [ ]:
@triton.jit
def m5_scalar_scale_kernel(x_ptr, out_ptr, scale_ptr, n_elements, BLOCK: tl.constexpr):
    pid = tl.program_id(0)
    offs = pid * BLOCK + tl.arange(0, BLOCK)
    mask = offs < n_elements
    scale = tl.load(scale_ptr)                 # shape ()
    x = tl.load(x_ptr + offs, mask=mask, other=0.0)
    tl.store(out_ptr + offs, x * scale, mask=mask)


def scalar_scale(x, scale, BLOCK=4):
    out = torch.empty_like(x)
    s = torch.tensor([scale], dtype=x.dtype, device=x.device)
    m5_scalar_scale_kernel[(triton.cdiv(x.numel(), BLOCK),)](
        x, out, s, x.numel(), BLOCK=BLOCK
    )
    return out


def test_scalar_scale() -> None:
    require_gpu("m5 scalar load")
    x = torch.arange(10.0, device=DEVICE)
    check("scalar broadcast scale", scalar_scale(x, 2.5, BLOCK=4), x * 2.5)


run_if_gpu(test_scalar_scale)

In [ ]:
@triton.jit
def m5_add_kernel(a_ptr, b_ptr, out_ptr, n_elements, BLOCK: tl.constexpr):
    pid = tl.program_id(0)
    offs = pid * BLOCK + tl.arange(0, BLOCK)
    mask = offs < n_elements
    a = tl.load(a_ptr + offs, mask=mask, other=0.0)
    b = tl.load(b_ptr + offs, mask=mask, other=0.0)
    tl.store(out_ptr + offs, a + b, mask=mask)


def add_tensors(a, b, BLOCK=4):
    out = torch.empty_like(a)
    m5_add_kernel[(triton.cdiv(a.numel(), BLOCK),)](
        a, b, out, a.numel(), BLOCK=BLOCK
    )
    return out


def test_add() -> None:
    require_gpu("m5 two-input add")
    torch.manual_seed(0)
    a = torch.randn(17, device=DEVICE)
    b = torch.randn(17, device=DEVICE)
    check("two inputs, one output", add_tensors(a, b, BLOCK=4), a + b)
    check(
        "two inputs, spine",
        add_tensors(torch.arange(10.0, device=DEVICE),
                    torch.ones(10, device=DEVICE), BLOCK=4),
        torch.arange(1.0, 11.0, device=DEVICE),
    )


run_if_gpu(test_add)

## Part IV — Method 03: read-modify-write and fused masked reductions

**Connection.** Builds on §5.13 (RMW) and §5.7 (neutral `other`).

**Rule.** RMW: `y` is input and output; use the same offsets and mask on both sides.
**Rule.** Fused reduction: load with the neutral `other`, reduce in registers, and store one scalar.

**Formula.**

$$
S=\sum_{i=0}^{P-1}\ \sum_{j=0}^{B-1}
\begin{cases}x_{iB+j}, & iB+j\lt N\\ 0, & iB+j\ge N\end{cases}
$$

**Formula.** For a max the inner identity is $-\infty$ instead of $0$.
**Where.** In-place updates and reductions that fit in one tile.
**Triton.** `tl.store(out_ptr + pid, value)` writes the per-program partial.
**Failure.** Sharing an address between programs makes the RMW a race; use one owner or `tl.atomic_add`.
**Bridge.** The next section differentiates the mask.

In [ ]:
@triton.jit
def m5_rmw_kernel(x_ptr, y_ptr, n_elements, BLOCK: tl.constexpr):
    pid = tl.program_id(0)
    offs = pid * BLOCK + tl.arange(0, BLOCK)
    mask = offs < n_elements
    x = tl.load(x_ptr + offs, mask=mask, other=0.0)
    y = tl.load(y_ptr + offs, mask=mask, other=0.0)
    tl.store(y_ptr + offs, y + x, mask=mask)


def add_inplace(x, y, BLOCK=4):
    m5_rmw_kernel[(triton.cdiv(x.numel(), BLOCK),)](x, y, x.numel(), BLOCK=BLOCK)
    return y


def test_rmw() -> None:
    require_gpu("m5 read-modify-write")
    x = torch.arange(1.0, 11.0, device=DEVICE)
    y = torch.arange(100.0, 110.0, device=DEVICE)
    before = y.clone()
    got = add_inplace(x, y, BLOCK=4)
    check("y += x in place", got, before + x)
    assert got is y and got.data_ptr() == y.data_ptr()
    print("in-place update verified")

    torch.manual_seed(0)
    x2 = torch.randn(17, device=DEVICE)
    y2 = torch.randn(17, device=DEVICE)
    ref = y2 + x2
    check("rmw, n=17", add_inplace(x2, y2.clone(), BLOCK=4), ref)


run_if_gpu(test_rmw)

In [ ]:
REDUCE_SUM, REDUCE_MAX = 0, 1


@triton.jit
def m5_reduce_kernel(x_ptr, out_ptr, n_elements,
                     MODE: tl.constexpr, BLOCK: tl.constexpr):
    pid = tl.program_id(0)
    offs = pid * BLOCK + tl.arange(0, BLOCK)
    mask = offs < n_elements
    if MODE == REDUCE_SUM:
        x = tl.load(x_ptr + offs, mask=mask, other=0.0)
        value = tl.sum(x, axis=0)
    else:
        x = tl.load(x_ptr + offs, mask=mask, other=-float("inf"))
        value = tl.max(x, axis=0)
    tl.store(out_ptr + pid, value)


def masked_reduce(x, mode=REDUCE_SUM, BLOCK=4):
    p = triton.cdiv(x.numel(), BLOCK)
    partial = torch.empty(p, dtype=x.dtype, device=x.device)
    m5_reduce_kernel[(p,)](x, partial, x.numel(), MODE=mode, BLOCK=BLOCK)
    total = partial.sum() if mode == REDUCE_SUM else partial.max()
    return total, partial


def test_reduce() -> None:
    require_gpu("m5 fused masked reduction")
    x = -torch.arange(1.0, 11.0, device=DEVICE)

    total, partial = masked_reduce(x, REDUCE_SUM, BLOCK=4)
    check("masked sum", total, torch.tensor(-55.0, device=DEVICE))
    check("masked sum vs torch", total, x.sum())
    assert partial.tolist() == [-10.0, -26.0, -19.0]

    best, partial = masked_reduce(x, REDUCE_MAX, BLOCK=4)
    check("masked max", best, torch.tensor(-1.0, device=DEVICE))
    check("masked max vs torch", best, x.max())
    assert partial.tolist() == [-1.0, -5.0, -9.0]
    assert max(partial.tolist() + [0.0]) == 0.0
    print("masked reductions verified against torch")


run_if_gpu(test_reduce)

## Forward and backward: the mask is a constant selector

**Connection.** Applies the §5.5 masked load to the derivative of a masked scale.

**Rule.** A fixed mask is a linear map on the active lanes.
**Formula.**

$$
y_i=\begin{cases}s\,x_i,&m_i=1\\ 0,&m_i=0\end{cases}
$$

**Formula.** $\partial L/\partial x_i=s\,m_i\,\partial L/\partial y_i$.
**Rule.** Masked-out lanes receive exactly zero gradient because the forward never read them.
**Where.** Backward passes of masked activations, dropout, and padded batches.
**Triton.** Save or recompute the predicate, not the loaded value.
**Failure.** Recomputing a different predicate between forward and backward routes the wrong gradient.
**Bridge.** The final sections explain the cost of the address pattern.

In [ ]:
x = torch.tensor([1.0, 2.0, 3.0, 4.0], requires_grad=True)
mask = torch.tensor([True, True, True, False])
y = torch.where(mask, 2.5 * x, torch.zeros_like(x))
y.sum().backward()
assert x.detach().tolist() == [1.0, 2.0, 3.0, 4.0]
assert mask.tolist() == [True, True, True, False]
assert y.detach().tolist() == [2.5, 5.0, 7.5, 0.0]
assert x.grad.tolist() == [2.5, 2.5, 2.5, 0.0]
assert (2.5 * mask.float()).tolist() == [2.5, 2.5, 2.5, 0.0]
assert x.grad[3].item() == 0.0

## Access patterns: contiguous, strided, and coalesced

**Connection.** Explains the cost side of §5.14.

**Rule.** The hardware groups the lanes of a warp into transactions; the count depends on the addresses.

**Formula.**

$$
\text{transactions}\approx\left\lceil\frac{\max p-\min p+b}{128}\right\rceil
$$

**Example.** For 32 contiguous `float32` lanes, $\max p-\min p=31\cdot4=124$ B, so the warp touches one 128 B line; a column access with row stride 32 puts the lanes 128 B apart, touches 32 lines, and uses only $1/32$ of the fetched bytes.
**Where.** Tile copies, transposes, and any `stride_m` / `stride_n` kernel.
**Triton.** Build the address expression so the last axis is contiguous.
**Failure.** A transposed copy reads the same elements and does the same arithmetic but pays more transactions.
**Bridge.** The pitfalls table collects the failures; the checklist tests the rules.

In [ ]:
B = torch.arange(32 * 32, dtype=torch.float32).reshape(32, 32)
line = 128
row_bytes = 32 * B.element_size()
col_step = B.stride(0) * B.element_size()
col_bytes = 31 * col_step + B.element_size()
assert B.shape == (32, 32) and B.stride() == (32, 1)
assert row_bytes == 128 and triton.cdiv(row_bytes, line) == 1
assert col_bytes == 3972 and triton.cdiv(col_bytes, line) == 32
assert row_bytes / (triton.cdiv(col_bytes, line) * line) == 0.03125

In [ ]:
@triton.jit
def m5_strided_copy_kernel(x_ptr, out_ptr, M, N, stride_m, stride_n,
                           BLOCK_M: tl.constexpr, BLOCK_N: tl.constexpr):
    pm = tl.program_id(0)
    pn = tl.program_id(1)
    rows = pm * BLOCK_M + tl.arange(0, BLOCK_M)
    cols = pn * BLOCK_N + tl.arange(0, BLOCK_N)
    mask = (rows[:, None] < M) & (cols[None, :] < N)
    src = rows[:, None] * stride_m + cols[None, :] * stride_n
    dst = rows[:, None] * N + cols[None, :]
    x = tl.load(x_ptr + src, mask=mask, other=0.0)
    tl.store(out_ptr + dst, x, mask=mask)


def strided_copy(x, BLOCK_M=2, BLOCK_N=2):
    M, N = x.shape
    sm, sn = x.stride()
    out = torch.empty((M, N), dtype=x.dtype, device=x.device)
    grid = (triton.cdiv(M, BLOCK_M), triton.cdiv(N, BLOCK_N))
    m5_strided_copy_kernel[grid](x, out, M, N, sm, sn,
                                 BLOCK_M=BLOCK_M, BLOCK_N=BLOCK_N)
    return out


def test_strided_copy() -> None:
    require_gpu("m5 strided copy")
    A = torch.arange(12.0, device=DEVICE).reshape(3, 4)
    assert A.stride() == (4, 1)
    check("row-major copy", strided_copy(A), A)
    assert A.T.stride() == (1, 4)
    check("column-major copy", strided_copy(A.T), A.T.contiguous())


run_if_gpu(test_strided_copy)

## Pitfalls

**Connection.** Collects the failure modes asserted in Part I.

**Rule.** Check mask, `other`, dtype, boundary, and shape before reading the output.

| Pitfall | What happens | Fix |
|---|---|---|
| mask only the store | the load still reads out of bounds | mask the load **and** the store |
| `other=0.0` for a max | a masked lane can win with zero | `other=-inf` |
| `other=0.0` for a log or a denominator | `log(0)=-inf`, `x/0=inf` | `other=1.0` |
| mask shape `(BLOCK,)` for a `(BM,BN)` tile | shape error or wrong lanes | `(rows[:,None]<M) & (cols[None,:]<N)` |
| `offs <= N` | one lane past the end is active | `offs < N` |
| `other=-inf` through an integer pointer | not representable | use a float dtype or `other=0` with a sum |
| RMW from two programs to one address | last write wins | one owner per address, or `tl.atomic_add` |
| passing a non-contiguous tensor without strides | the kernel reads the wrong elements | pass `stride_m, stride_n` or `.contiguous()` |
| storing a wider dtype through a narrow pointer | silent truncation | cast explicitly before `tl.store` |
| confusing `other` with a clamp | masked lanes are not clamped values | use `tl.minimum`/`tl.maximum` for clamping |
| `BLOCK` not a power of two | `tl.arange` rejects it | `triton.next_power_of_2` |
| tail sentinel confused with data | a stale value looks like a real element | compare only `out[:N]`, or check the sentinel explicitly |

## Self-check

**Connection.** Use these questions to test the rules from Part I without running the notebook.

1. Write the three rows of the $N=10$, `BLOCK=4` tile table from memory, including the mask.
2. What is `P` for $N=10$, `BLOCK=4`? How many declared lanes are there, and how many are out of bounds?
3. Why is `other=0.0` wrong for a max but right for a sum? Give the identity that each reduction needs.
4. Why is `other=0.0` dangerous for `log` and for a denominator? What value replaces it?
5. What does a masked store do to the destination lane: zero it, write `other`, or leave it?
6. A kernel loads with `mask` but stores without it. Which memory access is unsafe, and why is the error silent?
7. What is the 2-D mask for a tile at `pid_m=1`, `pid_n=1` with `M=3`, `N=5`, `BLOCK_M=2`, `BLOCK_N=4`? How many lanes are active?
8. Why does a column-major access pattern cost more transactions than a row-major one with the same element count?
9. What is the gradient of a masked scale with respect to a masked-out lane?
10. Can two programs safely increment the same address with `tl.store`? What should they use instead?

## Summary

**Thread recap.**
- `tl.load(ptr + offs, mask=..., other=...)` and `tl.store(ptr + offs, value, mask=...)` are the only two global-memory doors.
- The tile shape is compile-time (`BLOCK`); offsets, mask, and value must agree on it. `other` is a scalar broadcast to the tile.
- Compute the grid with $P=\lceil N/\text{BLOCK}\rceil$ and the tail mask with `offs < N`; for 2-D tiles, combine `rows < M` and `cols < N` with `&`.
- Mask both doors with the same predicate. A masked store leaves the old value untouched; it does not write zero.
- Choose `other` from the next operation: `0` for sum, `1` for product, `-inf` for max, `+inf` for min, `1` for log/denominator.
- Dtype sets the bytes moved and the interpretation of the value; cast before a narrow store and accumulate in `float32` when needed.
- Multiple pointers are just multiple arguments; each extra load or store adds traffic.
- Read-modify-write is safe when each address has one owner; cross-program accumulation needs atomics.
- Coalescing is about addresses: consecutive lanes should touch consecutive elements. Column-major access multiplies transactions.
- A mask is a constant selector in the backward pass: the gradient is masked by the same predicate.

**Where this leads.**
- Chapter 6 turns the 1-D offset map into 2-D and 3-D indexing.
- Continue with [Chapter 6 — Indexing and Offset Calculations](../ch06_indexing/06_complete.ipynb).

[Index](../../00_index.ipynb) · [Chapter README](README.md) · [← Previous](../ch04_pointers/04_complete.ipynb)

## Exercises

**Connection.** Each exercise breaks one rule from Part I.

1. **Mask table.** For $N=13$, `BLOCK=4`, write `P`, every `pid`, every `offs` row, and every mask. Which program has two active lanes?
2. **Wrong neutral.** Change the reduction kernel's max branch to `other=0.0` and find the smallest all-negative input that exposes the bug. Explain why the test input must be all negative.
3. **Store-only mask.** Delete the mask from the load in `m5_copy_kernel`, keep it on the store, and describe what the third program reads. Why does the test not catch it on every machine?
4. **2-D boundary.** Tile-copy a `(5,7)` matrix with `BLOCK_M=2`, `BLOCK_N=4`. List the four grid coordinates and the active-lane count for the corner tile.
5. **Sentinel audit.** Allocate the output with `fill=-7.0` and assert that exactly `P*BLOCK-N` lanes keep the sentinel. Generalise the assertion to any `N` and `BLOCK`.
6. **Dtype traffic.** For $N=2^{20}$, compute the bytes moved by a copy in `float32`, `float16`, and `int8`. How much bandwidth does each save relative to `float32`?
7. **Scalar reuse.** In `m5_scalar_scale_kernel`, replace the scalar load with a per-lane load of the same address. How many extra transactions does the tile issue?
8. **Two outputs.** Add an output pointer to the add kernel that stores `a*b`. Count the reads and writes per element before and after the change.
9. **RMW race.** Launch two programs that both read and write the same 16 elements and show that the result depends on the order. Then rewrite the update with `tl.atomic_add` and explain the new cost.
10. **Column-major copy.** Extend `strided_copy` to a non-square `(7,5)` tensor and check both `A` and `A.T.contiguous()` against the reference. Why must the destination use `N` and not `stride_n`?

In [ ]:
print("CPU tests ...")
test_worked_example()
test_reference()

print("\nGPU tests (skipped without CUDA/ROCm) ...")
for fn in (
    test_copy,
    test_scalar_scale,
    test_add,
    test_rmw,
    test_reduce,
    test_strided_copy,
):
    run_if_gpu(fn)